In [1]:
!pip install lion-pytorch

In [2]:
import os
import cv2
import gc
import sys
import random
import warnings
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score
from sklearn.preprocessing import LabelEncoder

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import autocast, GradScaler
from torchvision import models
from torch import autocast 
from PIL import Image
from lion_pytorch import Lion

# Albumentations for "Heavy" Augmentation
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Suppress the harmless scheduler warning
warnings.filterwarnings("ignore", category=UserWarning)

# ==========================================
# 1. CONFIGURATION
# ==========================================
class Config:
    SEED = 42
    IMG_SIZE = 384
    BATCH_SIZE = 16
    EPOCHS = 30
    LR = 1e-4
    WEIGHT_DECAY = 1e-2
    N_FOLDS = 5
    NUM_WORKERS = 2 # Lowered to avoid multiprocessing locks in some environments
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # --- PATHS ---
    BASE_PATH = "/kaggle/input/dataset/dataset"
    
    TRAIN_IMG_DIR = os.path.join(BASE_PATH, "images")
    TRAIN_MASK_DIR = os.path.join(BASE_PATH, "masks")
    TRAIN_LABELS_CSV = os.path.join(BASE_PATH, "train_labels.csv")

    TEST_ROOT = "/kaggle/input/test-data/test_data"
    TEST_IMG_DIR = os.path.join(TEST_ROOT, "test_img")
    TEST_MASK_DIR = os.path.join(TEST_ROOT, "test_mask")

    # --- ACCELERATOR PATHS ---
    FAST_TRAIN_IMG = "/kaggle/working/fast_train/images"
    FAST_TRAIN_MASK = "/kaggle/working/fast_train/masks"
    FAST_TEST_IMG = "/kaggle/working/fast_test/images"
    FAST_TEST_MASK = "/kaggle/working/fast_test/masks"

def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

seed_everything(Config.SEED)

In [3]:
# 2. UTILS: Reinhard & Smart Crop
# ==========================================
class ReinhardNormalizer:
    def __init__(self):
        self.target_mean = np.array([148.60, 169.30, 105.47])
        self.target_std = np.array([41.56, 11.23, 14.39])

    def __call__(self, img_rgb):
        try:
            img_lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB).astype("float32")
            img_mean = np.mean(img_lab, axis=(0,1))
            img_std = np.std(img_lab, axis=(0,1)) + 1e-5
            img_lab = (img_lab - img_mean) / img_std * self.target_std + self.target_mean
            img_lab = np.clip(img_lab, 0, 255).astype("uint8")
            return cv2.cvtColor(img_lab, cv2.COLOR_LAB2RGB)
        except:
            return img_rgb

def smart_square_crop_dual(img, mask):
    """
    Crops BOTH image and mask to the same square ROI based on the mask's content.
    """
    if np.sum(mask) == 0:
        h, w = img.shape[:2]
        sz = min(h, w)
        y = (h-sz)//2
        x = (w-sz)//2
        return img[y:y+sz, x:x+sz], mask[y:y+sz, x:x+sz]

    rows, cols = np.where(mask > 0)
    y_min, y_max = np.min(rows), np.max(rows)
    x_min, x_max = np.min(cols), np.max(cols)

    h_roi = y_max - y_min
    w_roi = x_max - x_min
    side = max(h_roi, w_roi)

    center_y = y_min + h_roi // 2
    center_x = x_min + w_roi // 2

    y_start = max(0, center_y - side // 2)
    y_end = min(img.shape[0], center_y + side // 2)
    x_start = max(0, center_x - side // 2)
    x_end = min(img.shape[1], center_x + side // 2)

    crop_img = img[y_start:y_end, x_start:x_end]
    crop_mask = mask[y_start:y_end, x_start:x_end]

    # Pad to square if needed
    pad_h = side - crop_img.shape[0]
    pad_w = side - crop_img.shape[1]

    if pad_h > 0 or pad_w > 0:
        crop_img = cv2.copyMakeBorder(
            crop_img,
            top=pad_h//2, bottom=pad_h - pad_h//2,
            left=pad_w//2, right=pad_w - pad_w//2,
            borderType=cv2.BORDER_CONSTANT, value=[255, 255, 255]
        )
        crop_mask = cv2.copyMakeBorder(
            crop_mask,
            top=pad_h//2, bottom=pad_h - pad_h//2,
            left=pad_w//2, right=pad_w - pad_w//2,
            borderType=cv2.BORDER_CONSTANT, value=0
        )

    return crop_img, crop_mask

# ==========================================
# 3. ACCELERATOR: PRE-COMPUTE & SAVE
# ==========================================
def extract_basename(filename: str):
    return os.path.splitext(filename)[0].replace("img_", "").replace("img", "")

    
def preprocess_dual_and_save(df, img_dir, mask_dir, out_img_dir, out_mask_dir, is_test=False):
    os.makedirs(out_img_dir, exist_ok=True)
    os.makedirs(out_mask_dir, exist_ok=True)

    normalizer = ReinhardNormalizer()
    valid_files = []

    print(f"⚡ Optimizing data to {out_img_dir}...")

    for idx, row in tqdm(df.iterrows(), total=len(df)):
      # --- Filename logic ---
        filename = row['filename'] if is_test else row.get('image_name', row.get('sample_index', row[0]))
        base_id = extract_basename(filename)

        img_name = f"img_{base_id}.png"
        mask_name = f"mask_{base_id}.png"

        img_path = os.path.join(img_dir, img_name)
        mask_path = os.path.join(mask_dir, mask_name)

        save_img_path = os.path.join(out_img_dir, img_name)
        save_mask_path = os.path.join(out_mask_dir, mask_name)

        #print(f"DEBUG: Processing {base_id} -> {img_name}, {mask_name}")
        #print(f"DEBUG: img_path='{img_path}', mask_path='{mask_path}'")

        if os.path.exists(save_img_path) and os.path.exists(save_mask_path):
            valid_files.append(img_name)
            print(f"DEBUG: Already processed, skipping {base_id}")
            continue

        try:
            img = cv2.imread(img_path)
            if img is None:
                print(f"❌ Missing image at {img_path}")
                continue
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

            mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            if mask is None:
                mask = np.zeros((img.shape[0], img.shape[1]), dtype=np.uint8)

            img, mask = smart_square_crop_dual(img, mask)
            img = normalizer(img)

            img = cv2.resize(img, (Config.IMG_SIZE, Config.IMG_SIZE), interpolation=cv2.INTER_LANCZOS4)
            mask = cv2.resize(mask, (Config.IMG_SIZE, Config.IMG_SIZE), interpolation=cv2.INTER_NEAREST)

            Image.fromarray(img).save(save_img_path)
            Image.fromarray(mask).save(save_mask_path)
            valid_files.append(img_name)
            #print(f"✅ Saved {img_name} and {mask_name}")

        except Exception as e:
            print(f"❌ Exception processing {base_id}: {e}")
            continue

    return valid_files

In [4]:
# --- EXECUTE PREPROCESSING ---
df_train_raw = pd.read_csv(Config.TRAIN_LABELS_CSV)
if 'image_name' not in df_train_raw.columns:
    df_train_raw.rename(columns={'sample_index': 'image_name', 'filename': 'image_name'}, inplace=True)

LUMINAL = ["Luminal A", "Luminal B"]

df_train_raw["label_binary"] = df_train_raw["label"].apply(
    lambda x: 1 if x in LUMINAL else 0
)

NUM_CLASSES = 2
print("Binary classes: 0 = non-luminal, 1 = luminal")

# Run for Train
valid_train_files = preprocess_dual_and_save(
    df_train_raw, Config.TRAIN_IMG_DIR, Config.TRAIN_MASK_DIR, Config.FAST_TRAIN_IMG, Config.FAST_TRAIN_MASK, is_test=False
)
df_train_raw['processed_filename'] = df_train_raw['image_name'].apply(lambda x: x if x.endswith('.png') else x.split('.')[0]+'.png')
df_train = df_train_raw[df_train_raw['processed_filename'].isin(valid_train_files)].reset_index(drop=True)
print(f"Valid Train Images: {len(df_train)}")



Binary classes: 0 = non-luminal, 1 = luminal
⚡ Optimizing data to /kaggle/working/fast_train/images...


  0%|          | 0/581 [00:00<?, ?it/s]

DEBUG: Already processed, skipping 0000
DEBUG: Already processed, skipping 0001
DEBUG: Already processed, skipping 0002
DEBUG: Already processed, skipping 0003
DEBUG: Already processed, skipping 0004
DEBUG: Already processed, skipping 0005
DEBUG: Already processed, skipping 0006
DEBUG: Already processed, skipping 0007
DEBUG: Already processed, skipping 0008
DEBUG: Already processed, skipping 0009
DEBUG: Already processed, skipping 0010
DEBUG: Already processed, skipping 0011
DEBUG: Already processed, skipping 0012
DEBUG: Already processed, skipping 0013
DEBUG: Already processed, skipping 0014
DEBUG: Already processed, skipping 0015
DEBUG: Already processed, skipping 0016
DEBUG: Already processed, skipping 0017
DEBUG: Already processed, skipping 0018
DEBUG: Already processed, skipping 0019
DEBUG: Already processed, skipping 0020
DEBUG: Already processed, skipping 0021
DEBUG: Already processed, skipping 0022
DEBUG: Already processed, skipping 0023
DEBUG: Already processed, skipping 0024


/tmp/ipykernel_148/2567961811.py:87: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  filename = row['filename'] if is_test else row.get('image_name', row.get('sample_index', row[0]))


In [5]:
# DISTRIBUTION OF BINARY CLASSES
# ===============================

dist = df_train_raw["label_binary"].value_counts().sort_index()

# Counts
count_non_luminal = dist.get(0, 0)
count_luminal = dist.get(1, 0)

# Percentages
total = count_non_luminal + count_luminal
pct_non_luminal = (count_non_luminal / total) * 100 if total > 0 else 0
pct_luminal = (count_luminal / total) * 100 if total > 0 else 0

print("\n=== Binary Class Distribution (df_train_raw) ===")
print(f"Non-Luminal (0): {count_non_luminal} ({pct_non_luminal:.2f}%)")
print(f"Luminal (1):     {count_luminal} ({pct_luminal:.2f}%)")
print(f"Total: {total}")



=== Binary Class Distribution (df_train_raw) ===
Non-Luminal (0): 219 (37.69%)
Luminal (1):     362 (62.31%)
Total: 581


In [6]:
# # Run for Test
# test_files = os.listdir(Config.TEST_IMG_DIR)
# df_test_raw = pd.DataFrame({'filename': test_files})
# valid_test_files = preprocess_dual_and_save(
#     df_test_raw, Config.TEST_IMG_DIR, Config.TEST_MASK_DIR, Config.FAST_TEST_IMG, Config.FAST_TEST_MASK, is_test=True
# )
# df_test_raw['processed_filename'] = df_test_raw['filename'].apply(lambda x: x.split('.')[0]+'.png')
# df_test = df_test_raw[df_test_raw['processed_filename'].isin(valid_test_files)].reset_index(drop=True)
# print(f"Valid Test Images: {len(df_test)}")

⚡ Optimizing data to /kaggle/working/fast_test/images...


  0%|          | 0/477 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [7]:
# 4. DATASET & AUGMENTATIONS
# ==========================================
def find_best_threshold(y_true, y_prob):
    thresholds = np.linspace(0.2, 0.8, 61)
    f1s = []
    for t in thresholds:
        preds = (y_prob >= t).astype(int)
        f1s.append(f1_score(y_true, preds, average='macro'))
    best_idx = np.argmax(f1s)
    return thresholds[best_idx], f1s[best_idx]
    
class DualStreamDataset(Dataset):
    def __init__(self, df, img_dir, mask_dir, transforms=None, is_test=False):
        self.df = df
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.transforms = transforms
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        fname = row['processed_filename']
        mask_fname = fname.replace("img_", "mask_")  # mask_0000.png


        img_path = os.path.join(self.img_dir, fname)
        mask_path = os.path.join(self.mask_dir, mask_fname)


        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

        if self.transforms:
            augmented = self.transforms(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']

        if mask.ndim == 2:
            mask = mask.unsqueeze(0)
        mask = mask.float() / 255.0

        if self.is_test:
            # Handle possible naming variations for submission
            orig_name = row['processed_filename'] if self.is_test else row['image_name']
            return image, mask, orig_name

        return image, mask, torch.tensor(row['label_binary'], dtype=torch.long)


import torch
import torch.nn as nn
import torch.nn.functional as F

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=None, reduction="mean"):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha
        self.reduction = reduction

    def forward(self, logits, targets):
        ce_loss = F.cross_entropy(
            logits,
            targets,
            reduction="none",
            weight=self.alpha
        )

        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss

        if self.reduction == "mean":
            return focal_loss.mean()
        elif self.reduction == "sum":
            return focal_loss.sum()
        else:
            return focal_loss


def get_transforms(data='train'):
    if data == 'train':
        return A.Compose([
            A.Resize(Config.IMG_SIZE, Config.IMG_SIZE),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.GridDistortion(5, 0.1, p=0.3),

            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2()
        ])
    else:
        return A.Compose([
            A.Resize(Config.IMG_SIZE, Config.IMG_SIZE),
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2()
        ])

# ==========================================
# 5. MODEL: DUAL-STREAM TEMPLE
# ==========================================
class DualStreamModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        # --- Stream 1: RGB (Texture) ---
        self.rgb_backbone = models.efficientnet_b1(weights=models.EfficientNet_B1_Weights.DEFAULT)
        rgb_dim = self.rgb_backbone.classifier[1].in_features
        self.rgb_backbone.classifier = nn.Identity()

        # --- Stream 2: Mask (Geometry) ---
        self.mask_backbone = models.efficientnet_b1(weights=models.EfficientNet_B1_Weights.DEFAULT)
        # Modify first layer: 3 channels -> 1 channel
        orig_conv = self.mask_backbone.features[0][0]
        new_conv = nn.Conv2d(1, 32, kernel_size=3, stride=2, padding=1, bias=False)
        with torch.no_grad():
            new_conv.weight[:] = torch.mean(orig_conv.weight, dim=1, keepdim=True)
        self.mask_backbone.features[0][0] = new_conv
        mask_dim = self.mask_backbone.classifier[1].in_features
        self.mask_backbone.classifier = nn.Identity()

        # --- The Summit (Fusion) ---
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Sequential(
            nn.Linear(rgb_dim + mask_dim, 512),
            nn.BatchNorm1d(512),
            nn.SiLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, rgb, mask):
        feat_rgb = self.rgb_backbone(rgb)
        feat_mask = self.mask_backbone(mask)
        combined = torch.cat((feat_rgb, feat_mask), dim=1)
        combined = self.dropout(combined)
        return self.fc(combined)

# ==========================================
# 6. TRAINING ENGINE
# ==========================================

def train_fold(fold, train_idx, val_idx):
    print(f"\n================ FOLD {fold+1}/{Config.N_FOLDS} ================")

    window_size = 3
    f1_window = []
    best_smoothed_f1 = 0.0
    best_raw_f1 = 0.0

    train_data = df_train.iloc[train_idx].reset_index(drop=True)
    val_data = df_train.iloc[val_idx].reset_index(drop=True)

    # Weighted Sampler
    class_counts = train_data['label_binary'].value_counts().sort_index().values
    alpha = torch.tensor(class_counts.sum() / class_counts, dtype=torch.float, device=Config.DEVICE)
    alpha = alpha / alpha.sum() 


    train_ds = DualStreamDataset(train_data, Config.FAST_TRAIN_IMG, Config.FAST_TRAIN_MASK, transforms=get_transforms('train'))
    val_ds = DualStreamDataset(val_data, Config.FAST_TRAIN_IMG, Config.FAST_TRAIN_MASK, transforms=get_transforms('val'))

    train_loader = DataLoader(train_ds, batch_size=Config.BATCH_SIZE, shuffle = True, num_workers=Config.NUM_WORKERS, pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=Config.NUM_WORKERS, pin_memory=True)

    model = DualStreamModel(NUM_CLASSES).to(Config.DEVICE)
    optimizer = Lion(model.parameters(), lr=Config.LR, weight_decay=Config.WEIGHT_DECAY)

    scheduler = optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=1e-4, steps_per_epoch=len(train_loader), epochs=Config.EPOCHS, pct_start=0.3
    )
    criterion = FocalLoss(gamma=1.5, alpha=alpha)

    scaler = torch.amp.GradScaler(device="cuda")

    best_f1 = 0.0
    patience = 9
    counter = 0

    for epoch in range(Config.EPOCHS):
        model.train()
        train_loss = 0

        for rgb, mask, labels in train_loader:
            rgb = rgb.to(Config.DEVICE)
            mask = mask.to(Config.DEVICE)
            labels = labels.to(Config.DEVICE)

            optimizer.zero_grad()
            with torch.amp.autocast(device_type="cuda"): 
                out = model(rgb, mask)
                loss = criterion(out, labels)


            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            # Unconditional step to avoid PyTorch version conflicts
            scheduler.step()

            train_loss += loss.item()

        avg_loss = train_loss / len(train_loader)
        # Validation
        model.eval()
        probs_all = []
        targets_all = []
        
        with torch.no_grad():
            for rgb, mask, labels in val_loader:
                rgb = rgb.to(Config.DEVICE)
                mask = mask.to(Config.DEVICE)
        
                logits = model(rgb, mask)
        
                probs = torch.softmax(logits, dim=1)[:, 1]  # P(class=1)
                probs_all.extend(probs.cpu().numpy())
                targets_all.extend(labels.numpy())

        targets_all = np.array(targets_all)
        probs_all = np.array(probs_all)
        
        # ---------- Raw F1 @ 0.5 (for logging only)
        raw_preds = (probs_all >= 0.5).astype(int)
        raw_f1 = f1_score(targets_all, raw_preds, average="macro")
        
        # ---------- Threshold sweep (main metric)
        thresholds = np.linspace(0.3, 0.7, 41)
        f1s = []
        
        for t in thresholds:
            preds = (probs_all >= t).astype(int)
            f1s.append(f1_score(targets_all, preds, average="macro"))
        
        best_idx = np.argmax(f1s)
        best_thresh = thresholds[best_idx]
        best_thresh_f1 = f1s[best_idx]
        
        # ---------- Moving average on best-threshold F1
        f1_window.append(best_thresh_f1)
        if len(f1_window) > window_size:
            f1_window.pop(0)
        
        smoothed_f1 = np.mean(f1_window)
        
        print(
            f"Ep {epoch+1:02d} | "
            f"Loss: {train_loss/len(train_loader):.4f} | "
            f"Raw F1@0.5: {raw_f1:.4f} | "
            f"Best F1: {best_thresh_f1:.4f} (t={best_thresh:.2f}) | "
            f"Smoothed: {smoothed_f1:.4f} | "
            f"Best: {best_smoothed_f1:.4f}"
        )
        
        # =======================
        # Early stopping
        # =======================
        if smoothed_f1 > best_smoothed_f1:
            best_smoothed_f1 = smoothed_f1
            best_threshold = best_thresh  # optional: store per fold
            torch.save(model.state_dict(), f"dual_fold_{fold}.pth")
            counter = 0
        else:
            counter += 1
            if counter >= patience:
                print(f"🛑 Early stopping at epoch {epoch+1}")
                break

    return best_smoothed_f1, best_threshold  

---

In [ ]:
# --- RUN TRAIN ---
skf = StratifiedKFold(n_splits=Config.N_FOLDS, shuffle=True, random_state=Config.SEED)
fold_scores = []

fold_metrics = {}

for fold, (train_idx, val_idx) in enumerate(skf.split(df_train, df_train['label_binary'])):
    # train_fold should now return both best F1 and best threshold
    best_f1, best_threshold = train_fold(fold, train_idx, val_idx)
    
    # Save per-fold metrics
    fold_metrics[fold] = {
        'best_f1': best_f1,
        'best_threshold': best_threshold
    }

    fold_scores.append(best_f1)

print(f"\n==================================")
print(f"FINAL CV F1: {np.mean(fold_scores):.4f}")
print("Fold metrics:", fold_metrics)

print(f"==================================")



================ FOLD 1/5 ================
Ep 01 | Loss: 0.0729 | Raw F1@0.5: 0.3054 | Best F1: 0.5119 (t=0.45) | Smoothed: 0.5119 | Best: 0.0000


In [ ]:
# # 7. INFERENCE (Dual TTA)
# # ==========================================
# print("\nStarting Inference...")
# test_ds = DualStreamDataset(df_test, Config.FAST_TEST_IMG, Config.FAST_TEST_MASK, transforms=get_transforms('val'), is_test=True)
# test_loader = DataLoader(test_ds, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=Config.NUM_WORKERS)

# models_list = []
# for i in range(Config.N_FOLDS):
#     m = DualStreamModel(NUM_CLASSES).to(Config.DEVICE)
#     m.load_state_dict(torch.load(f"dual_fold_{i}.pth"))
#     m.eval()
#     models_list.append(m)

# final_preds, final_fnames = [], []

# with torch.no_grad():
#     for rgb, mask, fnames in tqdm(test_loader):
#         rgb = rgb.to(Config.DEVICE)
#         mask = mask.to(Config.DEVICE)

#         # Dual TTA
#         rgb_inputs = [rgb, torch.flip(rgb, [3]), torch.flip(rgb, [2]), torch.rot90(rgb, 1, [2, 3])]
#         mask_inputs = [mask, torch.flip(mask, [3]), torch.flip(mask, [2]), torch.rot90(mask, 1, [2, 3])]

#         avg_probs = torch.zeros((rgb.size(0), NUM_CLASSES)).to(Config.DEVICE)

#         for model in models_list:
#             for i in range(len(rgb_inputs)):
#                 out = model(rgb_inputs[i], mask_inputs[i])
#                 avg_probs += torch.softmax(out, dim=1)

#         avg_probs /= (len(models_list) * len(rgb_inputs))

#         preds = torch.argmax(avg_probs, dim=1).cpu().numpy()
#         #decoded = le.inverse_transform(preds)

#         final_preds.extend(preds)
#         final_fnames.extend(fnames)

# sub = pd.DataFrame({'sample_index': final_fnames, 'label': final_preds})
# sub.to_csv("submissionGMFusion.csv", index=False)
# print("Submission Saved!")